<center><h1 style="margin-bottom: 0px;">Machine Learning I (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">K-Nearest Neighbour Modification and Performance Benchmarking</h2></center>

#### <center> Filipe Zheng (202406753), Maiara Guedes (202407694), Sílvia Pinto (202405988) </center> <br>

In [ ]:
%pip install pandas numpy scikit-learn

In [75]:
import os
import math
from pandas import read_csv
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score

### **1. Algorithm Selection and Implementation** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Having the freedom to choose any of the algorithms we learned in the course lectures, we decided to work with K-Nearest Neighbour. Due to its simplicity, it gives us many options for modification and improvement, and allows us to employ meticulous attention to each detail of this project.
<p style="text-indent: 2em;">To implement K-NN (K-Nearest Neighbour), we must first comprehend how it works. When K-NN receives a new object to classify, it compares it to all the objects in the training set, which are already correctly classified. This comparison is done by measuring the distance between them: how many attributes do they have in common, and what is the difference between the attributes in which they diverge? Then, it chooses the k closest training objects ("neighbours") and checks each of their target classes. The most common class in those k neighbours is the class it will assign to the new object.
<p style="text-indent: 2em;">The following code shows our implementation. It defines a class which allows us to create and use many individual KNN classifiers - each of them can be "fed" a training set, upon which it will make predictions on new, unseen datasets. It is the unmodified, conventional version of the K-NN algorithm, which we explained above.</div>

In [4]:
class KNN_Classifier():
    def __init__(self,k=5,*,distance_function=None):                                               # Initializes a new classifier.
        self.k = k                                                                                 # k is the number of neighbours to consider.
        self.df = None                                                                             # The classifier has not yet been "fed" the data - it is empty.
        self.df_target = None                                                                      # The same goes for the target dataframe.
        self.distF = distance_function or KNN_Classifier.distance                                  # Function to calculate the distance.

    def fit(self,data_frame,data_frame_target):                                                    # Fits the dataframes into the classifier.
        self.df = data_frame.reset_index(drop=True)
        self.df_target = data_frame_target.reset_index(drop=True)

    def distance(row1,row2):                                                                       # Default distance function (can be replaced when initializing).
        Sum = 0.0
        for i,j in zip(row1,row2):
            if (isinstance(i,(float,int)) and not(math.isnan(i) or math.isnan(j))): Sum+=abs(i-j)  # For each numeric attribute, add the difference to the sum.
            else: Sum += i!=j                                                                      # For each categorical attribute, add 1 if different, else 0.
        return Sum                                                                                 # Return the distance.

    def predict_row(self, row):
        dists = [(i, self.distF(row, row0)) for i, row0 in enumerate(self.df.values)]
        knn = [self.df_target.iloc[i] for i, _ in sorted(dists, key=lambda x: x[1])[:self.k]]
        counter = {}
        for i in knn:
            if not i in counter:
                counter[i] = 0
            counter[i] += 1
        return max(counter, key=counter.get)                                                                           # Return the predicted value for our object.

    def predict(self,df):                                                                          # Function to predict the target class.
        return [self.predict_row(row) for _,row in df.iterrows()]                                  # Returns target predictions for each of the rows in the dataframe.

### **2. Performance Measurement - Before Modification** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Now, we will test this original K-NN implementation on the datasets. By measuring various statistics of its performance, we can then decide what kind of modifications we will apply on the algorithm in order to improve those measurements.
<p style="text-indent: 2em;">But first... being provided with 3 different types of data characteristics, we must reflect on which will be the most challenging and interesting to analyse for K-NN. Each of the dataset groups is oriented to a different type of "problematic" data. Respectively, groups 1 , 2 and 3 are significantly affected by noise/outliers, class imbalance and multiclass classification. Considering K-NN's algorithm, we can make the following remarks:

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **1.** *Because K-NN is a "lazy learner" - it does not have a strong mathematical foundation, since it simply relies on the proximity of data points - it can be very sensitive to noise and outliers. For small values of k, a few outliers can greatly affect the prediction of the algorithm (such as a few elements of one class inside a large agglomeration of elements of another class).*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *Class imbalance is definitely a major influence on K-NN. Since its classification relies on majority voting, if a class is under-represented, it might be chosen much less than it should - because the neighbourhood of our object is statistically more likely to be dominated by the most common class.*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *Finally, multiclass classification is also troublesome. With an increase in the number of classes, the decision boundaries become more and more complex and fragmented. Also, areas where many classes overlap become very difficult to classify, since small changes to k will probably make our prediction jump from class to class.*


<p style="text-indent: 2em;">Taking into account the previous points, it seems most likely to us that dataset group 2 will impose a greater challenge to our algorithm (but all of them will definitely have some level of impact on K-NN). A very disproportionate class imbalance might not cause a loss of accuracy; maybe it will become even higher, but that is simply because practically every new object is falling into the larger class, completely disregarding the minority class.
<p style="text-indent: 2em;">Let us test our hypothesis in a tangible way: we will now use our KNN Classifier in each dataset group and analyse various performance measures, to determine which data characteristic truly causes the most damage to our predictions.</div>


In [71]:
def process_dataset(file):

    df = read_csv(file, na_values=['None', 'nan', ''])

    """nulls_limit = 0.50     # Se tiver mais de 50% de nulos, a coluna é descartada
    total_lines = len(df)

    for col in list(df.columns):
        empty = df[col].isnull().sum()   # Quantidade de espaços vazios numa coluna

        if empty/total_lines > nulls_limit:
            df = df.drop(col, axis=1)    # Apaga a coluna
    """

    X = df.iloc[:, :-1]         # Divisão de dados e classes
    Y = df.iloc[:, -1]

    if Y.dtype == 'object':                  # Encoding da target (se for categórico)
        Y = Y.astype('category').cat.codes





    #X = X.apply(pd.to_numeric, errors='coerce')

    X_train, X_test, y_train, y_test = train_test_split(X, Y, random_state=0)    # Split Treino/Teste

    for col in X.columns:                   # Encoding no conjunto de treino
         if X[col].dtype == 'object' or str(X[col].dtype) == 'category':
            continue  # Não mudamos os categóricos
            """if X[col].isnull().sum() > 0:      # Se a coluna for texto, preenchemos os buracos com a moda
                moda = X[col].mode()[0]
                X[col] = X[col].fillna(moda)
            """
            X[col] = X[col].astype('category').cat.codes
            X.loc[X[col]==-1] = float('nan')
         else:
                """ # Transformar para uma distribuição normal
                # calcular média e std ignorando NaN
                mean = np.nanmean(X_train[col], axis=0)
                std = np.nanstd(X_train[col], axis=0)
                if std==0:std=1
                # aplicar standardization
                X_train[col] = (X_train[col] - mean) / std
                mean = np.nanmean(X_test[col], axis=0)
                std = np.nanstd(X_test[col], axis=0)
                if std==0:std=1
                X_test[col] = (X_test[col] - mean) / std
                """
                #Min Max Scaler
                if np.all(np.isnan(X_train[col])): continue
                Xmax = np.nanmax(X_train[col],axis=0)
                Xmin = np.nanmin(X_train[col],axis=0)
                X_train[col] = (X_train[col] - Xmin) / (Xmax-Xmin)
                Xmax = np.nanmax(X_test[col],axis=0)
                Xmin = np.nanmin(X_test[col],axis=0)
                X_test[col] = (X_test[col] - Xmin) / (Xmax-Xmin)


    """imputer = SimpleImputer(strategy='mean')            # Preencher valores em falta com a média
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)


    scaler = StandardScaler()   # Escalonamento
    X_train = scaler.fit_transform(X_train_imputed)
    X_test = scaler.transform(X_test_imputed)
    """
    #X_train = X_train.astype(float)
    #X_test = X_test.astype(float)



    return X_train, X_test, y_train, y_test


In [82]:
base = r"class_imbalance" # Pasta onde os ficheiros estão guardados

files_names = [
    "dataset_38_sick.csv",
    "dataset_311_oil_spill.csv",
    "dataset_312_scene.csv",
    "dataset_316_yeast_ml8.csv",
    "dataset_450_analcatdata_lawsuit.csv",
    "dataset_463_backache.csv",
    "dataset_757_meta.csv",
    "dataset_764_analcatdata_apnea3.csv",
    "dataset_765_analcatdata_apnea2.csv",
    "dataset_767_analcatdata_apnea1.csv",
    "dataset_865_analcatdata_neavote.csv",
    "dataset_867_visualizing_livestock.csv",
    "dataset_875_analcatdata_chlamydia.csv",
    "dataset_940_water-treatment.csv",
    "dataset_947_arsenic-male-bladder.csv",
    "dataset_949_arsenic-female-bladder.csv",
    "dataset_950_arsenic-female-lung.csv",
    "dataset_951_arsenic-male-lung.csv",
    "dataset_954_spectrometer.csv",
    "dataset_958_segment.csv",
    "dataset_962_mfeat-morphological.csv",
    "dataset_966_analcatdata_halloffame.csv",
    "dataset_968_analcatdata_birthday.csv",
    "dataset_971_mfeat-fourier.csv",
    "dataset_976_JapaneseVowels.csv",
    "dataset_978_mfeat-factors.csv",
    "dataset_980_optdigits.csv",
    "dataset_984_analcatdata_draft.csv",
    "dataset_987_collins.csv",
    "dataset_995_mfeat-zernike.csv",
    "dataset_1000_hypothyroid.csv",
    "dataset_1002_ipums_la_98-small.csv",
    "dataset_1004_synthetic_control.csv",
    "dataset_1013_analcatdata_challenger.csv",
    "dataset_1014_analcatdata_dmft.csv",
    "dataset_1016_vowel.csv",
    "dataset_1018_ipums_la_99-small.csv",
    "dataset_1020_mfeat-karhunen.csv",
    "dataset_1021_page-blocks.csv",
    "dataset_1022_mfeat-pixel.csv",
    "dataset_1023_soybean.csv",
    "dataset_1039_hiva_agnostic.csv",
    "dataset_1045_kc1-top5.csv",
    "dataset_1049_pc4.csv",
    "dataset_1050_pc3.csv",
    "dataset_1056_mc1.csv",
    "dataset_1059_ar1.csv",
    "dataset_1061_ar4.csv",
    "dataset_1064_ar6.csv",
    "dataset_1065_kc3.csv"
]

files = [os.path.join(base, name) for name in files_names]  # Cria os caminhos completos

accuracies = []
f1_scores = []
for f in files:

    X_train, X_test, y_train, y_test = process_dataset(f)

    # Treinar o KNN
    modelo = KNN_Classifier(k=3)
    modelo.fit(X_train, y_train)


    prevision = modelo.predict(X_test)

    # Calcular a Accuracy

    accuracy = accuracy_score(y_test, prevision)
    F1_score = f1_score(y_test, prevision)
    accuracies.append(accuracy)
    f1_scores.append(F1_score)
    print(f"Dataset:{f}  Accuracy: {accuracy * 100:.2f}%   F1 score:{F1_score*100:.2f}%")

Dataset:class_imbalance/dataset_38_sick.csv  Accuracy: 92.15%   F1 score:41.27%
Dataset:class_imbalance/dataset_311_oil_spill.csv  Accuracy: 96.17%   F1 score:47.06%
Dataset:class_imbalance/dataset_312_scene.csv  Accuracy: 90.70%   F1 score:75.22%
Dataset:class_imbalance/dataset_316_yeast_ml8.csv  Accuracy: 98.51%   F1 score:0.00%
Dataset:class_imbalance/dataset_450_analcatdata_lawsuit.csv  Accuracy: 98.48%   F1 score:88.89%
Dataset:class_imbalance/dataset_463_backache.csv  Accuracy: 91.11%   F1 score:60.00%
Dataset:class_imbalance/dataset_757_meta.csv  Accuracy: 81.82%   F1 score:89.83%
Dataset:class_imbalance/dataset_764_analcatdata_apnea3.csv  Accuracy: 88.50%   F1 score:93.19%
Dataset:class_imbalance/dataset_765_analcatdata_apnea2.csv  Accuracy: 94.12%   F1 score:96.59%
Dataset:class_imbalance/dataset_767_analcatdata_apnea1.csv  Accuracy: 93.28%   F1 score:96.08%
Dataset:class_imbalance/dataset_865_analcatdata_neavote.csv  Accuracy: 64.00%   F1 score:0.00%
Dataset:class_imbalance/d

KeyboardInterrupt: 

In [78]:
#noise_outliers
path = "noise_outliers"
files_names = os.listdir(path)
files_noise = [os.path.join(path, name) for name in files_names]
accuracies_noise = []
f1_scores_noise = []
for f in files_noise:

    X_train, X_test, y_train, y_test = process_dataset(f)

    # Treinar o KNN
    modelo = KNN_Classifier(k=3)
    modelo.fit(X_train, y_train)


    prevision = modelo.predict(X_test)

    # Calcular a Accuracy
    accuracy = accuracy_score(y_test, prevision)
    F1_score = f1_score(y_test, prevision)
    accuracies_noise.append(accuracy)
    f1_scores_noise.append(F1_score)
    print(f"Dataset:{f}  Accuracy: {accuracy * 100:.2f}%   F1 Score: {F1_score*100:.2f}%")

Dataset:noise_outliers/dataset_450_analcatdata_lawsuit.csv  Accuracy: 98.48%   F1 Score: 88.89%
Dataset:noise_outliers/dataset_720_abalone.csv  Accuracy: 76.08%   F1 Score: 75.73%
Dataset:noise_outliers/dataset_335_monks-problems-3.csv  Accuracy: 92.81%   F1 Score: 93.33%
Dataset:noise_outliers/dataset_714_fruitfly.csv  Accuracy: 59.38%   F1 Score: 68.29%
Dataset:noise_outliers/dataset_466_schizo.csv  Accuracy: 67.06%   F1 Score: 58.82%
Dataset:noise_outliers/dataset_796_cpu.csv  Accuracy: 83.02%   F1 Score: 88.31%
Dataset:noise_outliers/dataset_747_servo.csv  Accuracy: 76.19%   F1 Score: 86.11%
Dataset:noise_outliers/dataset_164_molecular-biology_promoters.csv  Accuracy: 85.19%   F1 Score: 83.33%
Dataset:noise_outliers/dataset_481_biomed.csv  Accuracy: 96.23%   F1 Score: 97.14%
Dataset:noise_outliers/dataset_738_pharynx.csv  Accuracy: 79.59%   F1 Score: 83.87%
Dataset:noise_outliers/dataset_27_colic.csv  Accuracy: 81.52%   F1 Score: 84.96%
Dataset:noise_outliers/dataset_49_heart-c.csv

In [83]:
print("Class Imbalance:")
print(f"Size: {len(accuracies)}")
print(f"Mean Accuracy: {np.mean(accuracies)*100:.2f}\nMean F1 Score:{np.mean(f1_scores)*100:.2f}")
print("\nNoise/Outliers:")
print(f"Size: {len(accuracies_noise)}")
print(f"Mean Accuracy: {np.mean(accuracies_noise)*100:.2f}\nMean F1 Score:{np.mean(f1_scores_noise)*100:.2f}")

Class Imbalance:
Size: 36
Mean Accuracy: 91.57
Mean F1 Score:73.04

Noise/Outliers:
Size: 50
Mean Accuracy: 80.20
Mean F1 Score:77.52


In [ ]:
for file in files:
    df_check = read_csv(file, na_values=['None'])

    null_counts = df_check.isnull().sum()

    missing_data = null_counts[null_counts > 0]

    print("Quantidade de valores em falta por coluna:")
    print(missing_data)

    print(f"\nTotal de valores em falta no dataset: {df_check.isnull().sum().sum()}")

Quantidade de valores em falta por coluna:
age       1
sex     150
TSH     369
T3      769
TT4     231
T4U     387
FTI     385
TBG    3772
dtype: int64

Total de valores em falta no dataset: 6064
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quantidade de valores em falta por coluna:
correl      24
cancor2    240
fract2     240
dtype: int64

Total de valores em falta no dataset: 504
Quantidade de valores em falta por coluna:
Series([], dtype: int64)

Total de valores em falta no dataset: 0
Quanti